In [2]:
import torch
from model import MegaGPT

In [3]:
torch.manual_seed(0)
m = MegaGPT(context_size=64, embedding_size=128, nheads=4, nblocks=2).cuda().eval()
m.setup_cache(1, dtype=torch.float32)

toks = torch.randint(0, 50257, (1, 20), device="cuda")
with torch.no_grad():
    full = m(toks)                                    # no cache
    zero = torch.zeros((), dtype=torch.long, device="cuda")
    out = [m(toks[:, :10], pos=zero)]                 # prefill 10
    for i in range(10, 20):                           # decode 10 one at a time
        out.append(m(toks[:, i:i+1], pos=torch.tensor(i, device="cuda")))
    cached = torch.cat(out, dim=1)

print((full - cached).abs().max())  # ~1e-5 or smaller

AssertionError: Torch not compiled with CUDA enabled